# Spatial Analysis: Agro-climatic Potential yields in Coca-Growing Areas

### Objective
This script assesses the agricultural potential of areas currently used for illegal coca cultivation in Colombia. Specifically, it extracts potential yields for Natural Rubber, Cocoa, Banana, and Sugar Cane for 1 km² grid cells where coca was detected in 2022. The analysis uses GAEZ v4 data for the 2011–2040 period under the RCP 2.6 (HadGEM2-ES) climate scenario.

### Workflow
1.  **Configuration:** Define spatial parameters (converting degrees to meters for the local latitude).
2.  **Preprocessing:** Reproject and clip global yield rasters to the local UTM grid for multiple crops in batches.
3.  **Extraction:** Extract yield values for all crops to the centroid of each 1 km² grid cell.
4.  **Output:** Export a consolidated Excel table for statistical modeling and bioenergy potential calculation.

### Technical Configuration
* **Software Requirement:** ArcGIS Pro v3.3.0 (with Spatial Analyst Extension).
* **Coordinate System:** WGS 84 / UTM Zone 18N (EPSG: 32618).
* **Climate Scenario:** Time period 2011/2040, RCP 2.6, HadGEM2-ES, Rainfed, High Input level.
* **Inputs:** * `prub200b_yld.tif` (Natural Rubber)
    * `coco200b_ylf.tif` (Cocoa)
    * `bana200b_yld.tif` (Banana)
    * `SugarCane_PotentialYield_RCP2.6_HAD.tif` (Sugar Cane)
    (Source: https://gaez.fao.org/).
    * `Grillas1k_Coca_2022`: 1km vector grid of coca density (Source: https://www.biesimci.org/)

In [3]:
import arcpy
from arcpy.sa import *
import math

# ===================================================================================
# 🟢 USER ACTION REQUIRED: DEFINE YOUR PATHS HERE
# ===================================================================================

# 1. Where is your Geodatabase? (This will be used for all intermediate processing)
workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"

# 2. Where are your Input Rasters? (The global GAEZ yield TIF files)
rubber_raster = r"M:\Documents\Spatial_Analysis_Project\data\raw\prub200b_yld.tif"
cocoa_raster  = r"M:\Documents\Spatial_Analysis_Project\data\raw\coco200b_yld.tif"
banana_raster = r"M:\Documents\Spatial_Analysis_Project\data\raw\bana200b_yld.tif"
sugar_raster  = r"M:\Documents\Spatial_Analysis_Project\data\raw\SugarCane_PotentialYield_RCP2.6_HAD.tif"

# 3. Where is your Shapefile? (The Coca_1km_2022 grid)
shapefile = r"M:\Documents\Spatial_Analysis_Project\data\raw\Grillas1k_Coca_2022\Grillas1k.shp"

# 4. Where do you want the final Excel file?
output_excel = r"M:\Documents\Spatial_Analysis_Project\data\interim\Agroclimatic_Potential_Yields.xlsx"

# ===================================================================================
# END OF USER CONFIGURATION
# ===================================================================================

# Set Environment
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True
print(f"Workspace active: {workspace}")

Workspace active: M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb


## 1. Spatial Resolution Configuration
In this step, we define the spatial parameters for the analysis. The Area of Interest (AOI) is the density map of illegal coca cultivation for the year 2022 in Colombia reprensented in 1 km2 and 5 km2 grid cells , Colombia is located near the equator, requiring a conversion of the source raster resolution from degrees to meters to ensure accurate area calculations.

**Parameters:**
* **Latitude:** ~4.0° N (Centroid of Colombia).
* **Source Resolution:** 0.08333 degrees  Cell size X and Cell size Y
* **Goal:** Calculate the equivalent cell size in meters to prepare for the projection.

In [4]:
# Constants for conversion
average_latitude = 4.0  # Centroid of Colombia
degree_to_meter_equator = 111000
original_cell_size_degrees = 0.08333

# Calculate conversion factor based on latitude
conversion_factor = degree_to_meter_equator * math.cos(math.radians(average_latitude))
cell_size_meters = original_cell_size_degrees * conversion_factor

print(f"Original Resolution: {original_cell_size_degrees} degrees")
print(f"Calculated Resolution: {cell_size_meters:.2f} meters")

# Define Output Coordinate System: WGS 84 / UTM zone 18N (EPSG: 32618)
out_coordinate_system = arcpy.SpatialReference(32618)

Original Resolution: 0.08333 degrees
Calculated Resolution: 9227.10 meters


## 2. Reprojection and Clipping

We perform two critical spatial operations for each of the four agro-climatic potential datasets:
1.  **Project:** Transform the global GAEZ rasters to UTM Zone 18N using **NEAREST** neighbor resampling. This ensures that the potential yield values remain unchanged by interpolation during the coordinate transformation.
2.  **Clip:** Cut the rasters to the specific bounding box of the Coca 2022 grid. This limits the data to the study area and significantly optimizes processing speed for the subsequent extraction steps.

In [5]:
# List of rasters to process based on the paths defined in Block 1
rasters_to_process = {
    "Rubber": rubber_raster,
    "Cocoa": cocoa_raster,
    "Banana": banana_raster,
    "SugarCane": sugar_raster
}

# Dictionary to store the paths of clipped rasters for the next step
final_clipped_rasters = {}

# Check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

try:
    # Get extent from the shapefile dynamically for a single consistent clip
    desc = arcpy.Describe(shapefile)
    extent = f"{desc.extent.XMin} {desc.extent.YMin} {desc.extent.XMax} {desc.extent.YMax}"

    for crop_name, crop_path in rasters_to_process.items():
        print(f"--- Processing {crop_name} Potential Yield ---")
        
        # Define intermediate output names within the workspace
        reproj_path = f"{workspace}\\{crop_name}_potential_reproj"
        clip_path = f"{workspace}\\{crop_name}_potential_clip"

        # Step 1: Projecting raster
        print(f"Step 1: Projecting {crop_name}...")
        arcpy.management.ProjectRaster(
            crop_path, 
            reproj_path, 
            out_coordinate_system, 
            resampling_type="NEAREST", 
            cell_size=cell_size_meters
        )

        # Step 2: Clipping raster
        print(f"Step 2: Clipping {crop_name}...")
        arcpy.management.Clip(
            reproj_path, 
            extent, 
            clip_path, 
            "#", 
            -9999999, 
            "NONE", 
            "NO_MAINTAIN_EXTENT"
        )
        
        # Store path for the extraction block
        final_clipped_rasters[crop_name] = clip_path

    print("✅ Preprocessing complete for all crop datasets.")

except Exception as e:
    print(f"❌ Error during preprocessing: {e}")

--- Processing Rubber Potential Yield ---
Step 1: Projecting Rubber...
Step 2: Clipping Rubber...
--- Processing Cocoa Potential Yield ---
Step 1: Projecting Cocoa...
Step 2: Clipping Cocoa...
--- Processing Banana Potential Yield ---
Step 1: Projecting Banana...
Step 2: Clipping Banana...
--- Processing SugarCane Potential Yield ---
Step 1: Projecting SugarCane...
Step 2: Clipping SugarCane...
✅ Preprocessing complete for all crop datasets.


## 3. Extract Values to Points

To link the yield data to the coca sites, we perform a point-based extraction:
1.  **Centroids:** We convert the 1 km² polygon grid into centroid points to represent each cell.
2.  **Extraction:** We sample all four agro-climatic rasters simultaneously at each point location. This creates a consolidated dataset containing potential yields for Rubber, Cocoa, Banana, and Sugar Cane for every grid cell.

In [6]:
# Define names for the centroid features
base_name = "Agroclimatic_Yield_Extract"
centroid_points = f"{workspace}\\{base_name}_centroids"

try:
    print("Step 1: Generating centroids from grid polygons...")
    arcpy.management.FeatureToPoint(shapefile, centroid_points, "CENTROID")

    print("Step 2: Extracting all crop raster values to points...")
    
    # We use the 'final_clipped_rasters' dictionary created in the previous block
    # to map each crop to a clear field name in the output table
    extraction_list = [
        [final_clipped_rasters["Rubber"], "Yield_Rubber"],
        [final_clipped_rasters["Cocoa"], "Yield_Cocoa"],
        [final_clipped_rasters["Banana"], "Yield_Banana"],
        [final_clipped_rasters["SugarCane"], "Yield_SugarCane"]
    ]
    
    # Extract Multi Values to Points samples all rasters into the same attribute table
    arcpy.sa.ExtractMultiValuesToPoints(centroid_points, extraction_list, "NONE")
    
    print("Extraction complete. Potential yield fields added to centroids.")

except Exception as e:
    print(f"Error during extraction: {e}")

Step 1: Generating centroids from grid polygons...
Step 2: Extracting all crop raster values to points...
Extraction complete. Potential yield fields added to centroids.


## 5. Data Export

The final step is to export the processed attribute table into a clean tabular format. This table now contains the unique grid IDs alongside the potential yields for all four analyzed crops (Rubber, Cocoa, Banana, and Sugar Cane).

**Workflow:**
1.  **Table to Table:** The point feature class with extracted values is saved as a standalone table in the Geodatabase for archival.
2.  **Table to Excel:** The final dataset is exported to an Excel workbook, which will be ingested by the subsequent Data Analysis scripts in Python.

In [12]:
# Define the name for the final table in the Geodatabase
output_table_name = "Agroclimatic_Yields_Final"
output_table = f"{workspace}\\{output_table_name}"

try:
    print(f"Step 1: Saving attribute table to Geodatabase as {output_table_name}...")
    # 'centroid_points' contains the yields for all 4 crops from the previous block
    arcpy.conversion.TableToTable(centroid_points, workspace, output_table_name)
    
    # Step 2: Export to the final Excel path defined in Block 1
    print(f"Step 2: Exporting to Excel: {output_excel}")
    arcpy.conversion.TableToExcel(output_table, output_excel)
    
    print("✅ SUCCESS: Agro-climatic Spatial Analysis Finished. Data is ready for Phase 2.")

except Exception as e:
    print(f"Error during export: {e}")

finally:
    # Release the Spatial Analyst extension license
    arcpy.CheckInExtension("Spatial")

# ==============================================================================
# OUTPUT: This script generates 'Agroclimatic_Potential_Yields.xlsx'.
# This file contains the following fields per grid cell:
# - Yield_Rubber: Potential yield for Natural Rubber.
# - Yield_Cocoa: Potential yield for Cocoa.
# - Yield_Banana: Potential yield for Banana.
# - Yield_SugarCane: Potential yield for Bioenergy (Sugar Cane).
# ==============================================================================

Step 1: Saving attribute table to Geodatabase as Agroclimatic_Yields_Final...
Step 2: Exporting to Excel: M:\Documents\Spatial_Analysis_Project\data\interim\Agroclimatic_Potential_Yields.xlsx
✅ SUCCESS: Agro-climatic Spatial Analysis Finished. Data is ready for Phase 2.


## 6. Uncertainty Analysis: Intra-cell Variability

### Objective
To quantify sub-grid heterogeneity and spatial uncertainty within the GAEZ yield data, we use a **cross-point sampling method**. This approach measures the internal variability of potential yields within each 1 km² grid cell.

### Methodology
1.  **Geometric Offsets:** From the geometric centroid of each cell, we generate four equidistant sampling points (North, South, East, West) at a distance of 0.5 km. This ensures the samples represent the cell's "corners" while remaining within the 1 km² boundary.
2.  **Sampling:** Potential yield values for all analyzed crops are extracted at these four specific coordinates.
3.  **Standard Deviation:** The standard deviation ($\sigma$) of these four values is calculated during the statistical phase to provide a quantitative measure of intra-cell spatial variability.

**Equation:**
$$\sigma = \sqrt{\frac{\sum_{i=1}^{4} (x_i - \bar{x})^2}{4}}$$

Where:
* $x_i$: Yield value at cross-point $i$.
* $\bar{x}$: Mean value of the four cross-points.

In [13]:
# ===================================================================================
# 🟢 USER ACTION REQUIRED: IDENTIFY EXISTING CROSS-POINTS
# ===================================================================================
# . Project Workspace
workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"

# Input: Existing cross-points (Assuming they were created in the NR Analysis)
# If running standalone, ensure this feature class exists.
existing_cross_points = f"{workspace}\\Coca_2022_Cross_Points"

# Output: Statistics Table
stats_table = f"{workspace}\\Agroclimatic_Variability_Stats"

# Grouping Field
join_field_id = "GRI_1K_F_1"

try:
    arcpy.CheckOutExtension("Spatial")
    
    print("Step 1: Extracting Agroclimatic raster values to cross-points...")
    # Define extraction mapping: [Raster, Field_Name]
    # We use the clipped rasters defined in Block 3
    extraction_list_var = [
        [final_clipped_rasters["Rubber"], "Var_Rubber"],
        [final_clipped_rasters["Cocoa"], "Var_Cocoa"],
        [final_clipped_rasters["Banana"], "Var_Banana"],
        [final_clipped_rasters["SugarCane"], "Var_SugarCane"]
    ]
    
    # Extract values directly into the existing feature class
    # Note: This appends fields Var_Rubber, Var_Cocoa, etc. to the cross-points
    arcpy.sa.ExtractMultiValuesToPoints(existing_cross_points, extraction_list_var, "NONE")
    
    print("Step 2: Calculating Standard Deviation for all crops...")
    # Calculate STD for each crop field, grouped by Grid ID
    stats_fields = [
        ["Var_Rubber", "STD"],
        ["Var_Cocoa", "STD"],
        ["Var_Banana", "STD"],
        ["Var_SugarCane", "STD"]
    ]
    
    arcpy.analysis.Statistics(
        in_table=existing_cross_points, 
        out_table=stats_table, 
        statistics_fields=stats_fields, 
        case_field=join_field_id
    )
    
    print(f"✅ Statistics calculated. Table created at: {stats_table}")

except Exception as e:
    print(f"❌ Error during variability calculation: {e}")

finally:
    arcpy.CheckInExtension("Spatial")

Step 1: Extracting Agroclimatic raster values to cross-points...
Step 2: Calculating Standard Deviation for all crops...
✅ Statistics calculated. Table created at: M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb\Agroclimatic_Variability_Stats


In [14]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

# 1. Inputs (Defined in previous steps)
# The table containing the Standard Deviations we just calculated
stats_table = f"{workspace}\\Agroclimatic_Variability_Stats"

# The Centroids containing the Mean Yields (Created in Block 4)
centroid_points = f"{workspace}\\Agroclimatic_Yield_Extract_centroids"

# 2. Output Excel Path
final_integrated_excel = r"M:\Documents\Spatial_Analysis_Project\data\interim\Agroclimatic_Potential_Yields_Final.xlsx"

# 3. Join Key
join_field_id = "GRI_1K_F_1"

# ==============================================================================
# PROCESSING
# ==============================================================================

try:
    print(f"Step 1: Joining Variability Statistics to Centroids...")
    
    # The fields created by the Statistics tool usually follow the format: STD_FieldName
    fields_to_join = ["STD_Var_Rubber", "STD_Var_Cocoa", "STD_Var_Banana", "STD_Var_SugarCane"]
    
    # Join the statistics table to the centroids
    arcpy.management.JoinField(
        in_data=centroid_points, 
        in_field=join_field_id, 
        join_table=stats_table, 
        join_field=join_field_id, 
        fields=fields_to_join
    )
    
    # Optional: Rename fields for a cleaner Excel header
    # We rename 'STD_Var_Rubber' -> 'STD_Rubber' for clarity
    print("Step 2: Formatting field names...")
    rename_map = {
        "STD_Var_Rubber": "STD_Rubber",
        "STD_Var_Cocoa": "STD_Cocoa",
        "STD_Var_Banana": "STD_Banana",
        "STD_Var_SugarCane": "STD_SugarCane"
    }
    
    for old_name, new_name in rename_map.items():
        try:
            arcpy.management.AlterField(centroid_points, old_name, new_name, new_name)
        except:
            pass # Ignore errors if field is locked or already renamed

    # --- Final Export ---
    print(f"Step 3: Exporting final integrated table to: {final_integrated_excel}")
    arcpy.conversion.TableToExcel(centroid_points, final_integrated_excel)
    
    print("✅ SUCCESS: Integrated Agro-climatic dataset exported.")
    print(f"Output saved at: {final_integrated_excel}")

except Exception as e:
    print(f"❌ Error during export: {e}")

Step 1: Joining Variability Statistics to Centroids...
Step 2: Formatting field names...
Step 3: Exporting final integrated table to: M:\Documents\Spatial_Analysis_Project\data\interim\Agroclimatic_Potential_Yields_Final.xlsx
✅ SUCCESS: Integrated Agro-climatic dataset exported.
Output saved at: M:\Documents\Spatial_Analysis_Project\data\interim\Agroclimatic_Potential_Yields_Final.xlsx
